# Random Forests - Bagging, OOB Intuition, and Feature Importance

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/mgmt_474_ai_logo_02-modified.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>MGMT47400 Predictive Analytics</center>
# <center>Professor: Davi Moreira </center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/blob/main/notebooks/12_random_forests_importance_student.ipynb)

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Explain bagging and why forests reduce variance
2. Train a random forest and tune the most impactful knobs
3. Use permutation importance responsibly
4. Compare forest vs tree vs linear/logistic baselines
5. Produce project-ready model comparison tables

---

> **📋 Participation Reminder:** This notebook contains **2 PAUSE-AND-DO exercises**. You are expected to complete all exercises before submitting your notebook.

---

## 💼 Why This Matters: Wisdom of the Crowd

The single decision tree for the **State Health Department** is interpretable but fragile — small changes in the data produce a completely different tree. The hospital board asks: *"Can we get both interpretability and stability?"*

Random forests combine hundreds of trees, each trained on a different bootstrap sample with random feature subsets. No single tree dominates; the forest votes. And a key bonus: by measuring how much each feature contributes across all trees, you can rank the measurements that matter most for diagnosis — "worst perimeter" and "worst concave points" emerge as top predictors.

> **Today's focus:** Building random forests for robust breast cancer classification, and using feature importance to identify the most diagnostically relevant cell measurements.

---

In [ ]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.precision', 4)
RANDOM_SEED = 474
np.random.seed(RANDOM_SEED)
print("✓ Setup complete!")

**Reading the output:**

The setup cell loads `RandomForestClassifier` from scikit-learn's ensemble module — the algorithm that will stabilize the fragile single tree from the previous notebook. `permutation_importance` from `sklearn.inspection` is the model-agnostic method we will use later to identify which cell-nucleus measurements (e.g., `worst_concave_points`, `worst_radius`) actually drive the screening diagnosis, as opposed to features that merely appear in many tree splits by coincidence.

The confirmation message `Setup complete!` with **RANDOM_SEED = 474** ensures reproducibility. All forest randomness — bootstrap sampling of patients, random feature subsets at each split — flows from this single seed, so the Health Department team can rerun the notebook next quarter and get identical results.

**Key takeaway:** Setting `n_jobs=-1` later in the forest constructor will parallelize tree training across all CPU cores. This matters because forests train 100-300 independent trees; without parallelization, training time scales linearly with the number of trees.

---

## 1. From Single Tree to Forest: The Bagging Idea

### The Variance Problem with Single Trees

The previous notebook exposed a critical weakness: the Health Department's depth-3 decision tree changes its root split when even a handful of training patients are swapped out. A model that gives different screening rules depending on which 398 patients happened to be in the training set is not one the clinical advisory board will approve for statewide deployment.

### The Solution: Bootstrap Aggregating (Bagging)

**Algorithm:**
1. Create B bootstrap samples (random sampling with replacement from the 398 training patients)
2. Train one decision tree on each bootstrap sample — each tree sees a slightly different patient mix
3. Aggregate predictions: majority vote for classification (malignant/benign), average for regression

**Why it works:**
- Averaging reduces variance — individual tree quirks cancel out
- Each tree sees ~63% of the training data; the other ~37% is "out-of-bag" (free validation)
- Errors from one tree are unlikely to be shared by all trees

### Random Forest = Bagging + Random Feature Selection

**Extra randomness:** At each split, the tree considers only a random subset of the 30 cell-nucleus features (typically `sqrt(30) ≈ 5-6`). This prevents `worst_radius` from dominating every tree's root split, forcing trees to discover alternative diagnostic paths through features like `mean_concavity` or `worst_texture`. The result is a more diverse ensemble where trees make *different* mistakes — and different mistakes average out.

One oncologist may diagnose differently on different days — high variance from a single decision-maker. A tumor board of 100 oncologists voting reduces that variance. Random forests apply the same principle: many diverse trees, each seeing a different patient sample and a different feature subset, voting together for more stable screening decisions.

> 💡 **Gemini Prompt:** "Load breast cancer dataset, split 70/30 with stratification (seed 474), and print set sizes and feature count."
>
> **After running, verify:**
> - Train/test sizes reflect 70/30 split
> - 30 features (breast cancer)
> - Split uses stratify=y
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Load data
data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=RANDOM_SEED, stratify=y)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")
print(f"Features: {X.shape[1]}")

**Reading the output:**

The breast cancer dataset is split into **398 training** and **171 test** tissue samples with stratification, preserving the roughly 63/37 benign-to-malignant class balance in each split — the same proportions the screening tool would encounter in a real clinical population. The dataset has **30 features**, all continuous measurements of cell nuclei (mean, standard error, and worst-case values for 10 properties like radius, texture, concavity, and symmetry).

**Why 30 features matters for Random Forests:** The algorithm randomly selects a subset of features at each split. With 30 features and the default `max_features='sqrt'`, each split considers roughly 5-6 candidate measurements, creating diversity among the trees. One tree might split on `worst_radius` at the root; another might split on `mean_concavity`. This diversity is the engine of variance reduction.

---

## 2. Single Tree vs Random Forest

The core claim of Random Forests is that averaging many decorrelated trees produces a model with **lower variance** and **higher overall accuracy** than any single tree. The experiment below makes this concrete for the screening task: we train one decision tree (`max_depth=5`) and one forest of 100 depth-5 trees on the same breast cancer dataset, using the same 5-fold stratified CV.

Watch for two things in the output: (1) the forest's mean ROC-AUC should be higher, and (2) its standard deviation across folds should be smaller. A tighter spread means the forest delivers more consistent screening performance regardless of which patients end up in each fold — exactly the stability the Health Department needs before rolling the model out to ten partner hospitals with different patient populations.

> 💡 **Gemini Prompt:** "Compare a single DecisionTree(depth=5) vs RandomForest(100 trees, depth=5) using 5-fold stratified CV with ROC-AUC. Print fold scores, mean improvement, variance reduction, and create a box plot."
>
> **After running, verify:**
> - Both models use max_depth=5 for fair comparison
> - Forest shows higher mean AND lower std
> - Box plot has two boxes with distinct colors
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Compare single tree vs forest
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

# Single tree (tuned depth)
tree = DecisionTreeClassifier(max_depth=5, random_state=RANDOM_SEED)
tree_scores = cross_val_score(tree, X_train, y_train, cv=cv, scoring='roc_auc')

# Random forest
forest = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=RANDOM_SEED)
forest_scores = cross_val_score(forest, X_train, y_train, cv=cv, scoring='roc_auc')

print("=== SINGLE TREE VS RANDOM FOREST ===")
print(f"\nSingle Tree (depth=5):")
print(f"  CV ROC-AUC: {tree_scores.mean():.4f} ± {tree_scores.std():.4f}")
print(f"  Fold scores: {tree_scores.round(4)}")

print(f"\nRandom Forest (100 trees, depth=5):")
print(f"  CV ROC-AUC: {forest_scores.mean():.4f} ± {forest_scores.std():.4f}")
print(f"  Fold scores: {forest_scores.round(4)}")

print(f"\n=== IMPROVEMENT ===")
print(f"Mean improvement: {(forest_scores.mean() - tree_scores.mean()):.4f}")
print(f"Variance reduction: {(tree_scores.std() - forest_scores.std()):.4f}")

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))
positions = [1, 2]
bp = ax.boxplot([tree_scores, forest_scores], positions=positions, widths=0.6, patch_artist=True)
for patch, color in zip(bp['boxes'], ['lightblue', 'lightgreen']):
    patch.set_facecolor(color)
ax.set_xticklabels(['Single Tree', 'Random Forest'])
ax.set_ylabel('ROC-AUC Score')
ax.set_title('Variance Reduction: Single Tree vs Random Forest')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("\n💡 Forest has higher mean AND lower variance")
print("💡 More stable predictions across different data samples")

**Reading the output:**

The printout shows fold-level ROC-AUC scores for both models. The single tree's scores vary more across folds (higher standard deviation), while the forest's scores cluster tightly around a higher mean. Typical numbers: the single tree scores around **0.95-0.97** with std ~0.02, while the forest scores **0.98-0.99** with std ~0.01.

The box plot reinforces this visually. The forest's box is both **higher** (better median screening accuracy) and **narrower** (less variance across patient partitions) than the single tree's box. The "Improvement" line quantifies the mean lift, and the "Variance reduction" line shows how much tighter the forest's spread is.

For the Health Department, the narrower box is arguably more important than the higher median. A single tree that scores 0.99 on one hospital's patients and 0.93 on another's creates an inconsistent screening experience. The forest's tight spread means patients at every partner hospital receive comparably accurate screening — the kind of consistency required for a statewide program.

**Key takeaway:** This is the entire motivation for ensemble methods. A single tree is a high-variance estimator; averaging 100 trees with different bootstrap samples and random feature subsets dramatically stabilizes predictions without sacrificing accuracy.

---

## 3. Tuning Random Forests

### Most Important Hyperparameters

**n_estimators** (number of trees in the forest)
- More trees = better performance (usually), but diminishing returns after ~100-500
- Each additional tree adds training time proportionally
- The screening tool's deployment budget determines how many trees are practical

**max_features** (features considered per split)
- `sqrt(n_features)` for classification (default) — roughly 5-6 of the 30 cell measurements per split
- Lower values force more diversity: one tree splits on `worst_radius`, another on `mean_texture`
- Too low means individual trees are weak; too high means all trees look alike

**max_depth** (tree depth)
- Controls individual tree complexity
- `None` (grow until pure) is common for forests — individual overfitting is tamed by averaging

**min_samples_split** (minimum samples to split)
- Higher values produce simpler trees within the forest
- Prevents splits based on a handful of edge-case patients

> 💡 **Gemini Prompt:** "Sweep n_estimators=[10,25,50,100,200,300] for RandomForest, compute 5-fold CV ROC-AUC for each. Print results table and plot CV score with error bars vs number of trees."
>
> **After running, verify:**
> - Results table shows n_estimators, cv_mean, cv_std
> - Performance plateaus around 100-200 trees
> - Error bars decrease as more trees are added
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Effect of number of trees
n_estimators_range = [10, 25, 50, 100, 200, 300]
results = []

for n_est in n_estimators_range:
    rf = RandomForestClassifier(n_estimators=n_est, random_state=RANDOM_SEED, n_jobs=-1)
    scores = cross_val_score(rf, X_train, y_train, cv=cv, scoring='roc_auc')
    results.append({
        'n_estimators': n_est,
        'cv_mean': scores.mean(),
        'cv_std': scores.std()
    })

results_df = pd.DataFrame(results)
print("=== N_ESTIMATORS SWEEP ===")
print(results_df.to_string(index=False))

# Plot
plt.figure(figsize=(10, 6))
plt.errorbar(results_df['n_estimators'], results_df['cv_mean'], 
             yerr=results_df['cv_std'], marker='o', capsize=5, linewidth=2)
plt.xlabel('Number of Trees')
plt.ylabel('CV ROC-AUC')
plt.title('Random Forest: Effect of Number of Trees')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n💡 Performance plateaus after ~100-200 trees")
print("💡 Use more trees for final model, fewer for experimentation")

**Reading the output:**

The table shows how CV ROC-AUC changes as we grow the forest from 10 to 300 trees. Performance improves rapidly from 10 to 50 trees as the ensemble accumulates enough diverse perspectives on the screening data, then plateaus around 100-200 trees. Going from 200 to 300 trees typically adds less than **0.001** to the mean ROC-AUC — a margin that would never change a screening decision.

The error-bar plot makes the diminishing returns clear: the curve flattens well before 300 trees. Meanwhile, training time scales linearly with `n_estimators`, so there is a practical cost to adding more trees beyond the plateau — hospital IT teams care about how long the model takes to retrain on updated data.

Standard deviation also decreases slightly with more trees, because averaging over a larger ensemble further stabilizes the screening estimate. However, the reduction is marginal after ~100 trees.

**Key takeaway:** Use 100-200 trees for experimentation and hyperparameter tuning. For the Health Department's final production model, bumping to 500+ trees adds a tiny extra stability boost, but the real performance levers are `max_features` and `max_depth`, which we tune next.

---

## 📝 PAUSE-AND-DO Exercise 1 (5 minutes)

**Task:** Tune `n_estimators` and `max_features` minimally and report effects.

---

> 💡 **Gemini Prompt:** "Tune max_features for RandomForest(100 trees) over ['sqrt','log2',0.3,0.5,None] using 5-fold CV ROC-AUC. Print comparison table."
>
> **After running, verify:**
> - Five max_features options tested
> - Table shows max_features, cv_mean, cv_std
> - None (all features) typically performs differently from sqrt/log2
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# YOUR SOLUTION CODE HERE
# Hint: Use the Gemini prompt above for step-by-step guidance


### YOUR ANALYSIS:

**Effect of max_features:**  
[What did you observe? How does it affect performance?]

**Recommendation:**  
[Which value would you use in production? Why?]

---

## 4. Out-of-Bag (OOB) Score

### Free Cross-Validation

Each tree in the forest is trained on a bootstrap sample containing ~63% of the 398 training patients. The remaining ~37% — the "out-of-bag" samples — are patients that particular tree never saw during training. By aggregating OOB predictions across all 100 trees, the forest produces an honest performance estimate without requiring a separate validation set.

**OOB Score ≈ Cross-Validation Score**
- **Faster than CV:** Computed during training at zero extra cost (no additional model fits)
- **Uses all data for training:** Every patient contributes to some trees' training and other trees' validation
- **Good for initial screening:** Quick check before investing in the full 5-fold CV sweep

For the Health Department's 569-sample dataset, where every tissue sample is valuable, OOB scoring is especially attractive — it avoids the data-efficiency penalty of holding out a separate validation set.

> 💡 **Gemini Prompt:** "Train RandomForest with oob_score=True (100 trees, seed 474), compare OOB accuracy to 5-fold CV accuracy, print both and their difference."
>
> **After running, verify:**
> - OOB score from rf.oob_score_
> - CV uses 5-fold StratifiedKFold
> - Difference between OOB and CV is small (<0.02)
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Compare OOB vs CV
rf_oob = RandomForestClassifier(
    n_estimators=100,
    oob_score=True,
    random_state=RANDOM_SEED,
    n_jobs=-1
)

rf_oob.fit(X_train, y_train)
oob_score = rf_oob.oob_score_

# Compare to CV
cv_scores = cross_val_score(rf_oob, X_train, y_train, cv=cv, scoring='accuracy')

print("=== OOB VS CROSS-VALIDATION ===")
print(f"OOB Score (free): {oob_score:.4f}")
print(f"CV Score (5-fold): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"Difference: {abs(oob_score - cv_scores.mean()):.4f}")

print("\n💡 OOB score is a good proxy for test performance")
print("💡 Use OOB for quick iterations, CV for final evaluation")

**Reading the output:**

Two numbers are printed side by side: the **OOB score** (computed internally by the forest from out-of-bag patients) and the **5-fold CV score**. Both are accuracy estimates on the screening task. Typically they agree to within **0.005-0.01**, confirming that OOB is a reliable free proxy for cross-validation.

The OOB approach works because each tree is trained on only ~63% of the patients (a bootstrap sample), so the remaining ~37% serves as a built-in validation set for that tree. Aggregating these per-tree OOB predictions across all 100 trees yields an unbiased performance estimate — effectively a leave-37%-out validation computed for free during training.

**Why this matters for the screening project:** OOB scores are computed during training with zero extra cost, while 5-fold CV requires fitting 5 separate forests (5x the compute). For quick hyperparameter screening — "does `max_features=0.3` beat `sqrt`?" — OOB gives an immediate answer. Reserve full CV for the final model comparison where you need the most reliable numbers to present to the Health Department board.

---

## 5. Feature Importance

### Which Cell Measurements Drive the Diagnosis?

The Health Department's lab directors have a practical question: *"Which measurements should we invest in standardizing across partner hospitals?"* If `worst_concave_points` drives 20% of the model's decisions but `mean_symmetry` drives 0.5%, the lab should prioritize calibrating concavity measurements first. Feature importance answers this question.

**1. Gini/Entropy Importance (built-in)**
- Based on how much each feature reduces impurity across all splits in all trees
- Fast to compute — available immediately after training
- Biased toward high-cardinality continuous features and double-counts correlated measurements (e.g., `worst_radius` and `worst_perimeter`)

**2. Permutation Importance (recommended)**
- Shuffle one feature's values, measure how much screening performance drops
- More reliable because it measures actual predictive contribution, not just split frequency
- Slower to compute but worth the wait for production decisions

> 💡 **Gemini Prompt:** "Train a final RandomForest with 200 trees, extract Gini feature importances into a sorted DataFrame, print top 10, and create a horizontal bar chart of top 15."
>
> **After running, verify:**
> - Feature importances from rf_final.feature_importances_
> - Top 10 printed in descending order
> - Bar chart shows top 15 features
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Train final forest
rf_final = RandomForestClassifier(n_estimators=200, random_state=RANDOM_SEED, n_jobs=-1)
rf_final.fit(X_train, y_train)

# Built-in importance
builtin_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_final.feature_importances_
}).sort_values('importance', ascending=False)

print("=== BUILT-IN FEATURE IMPORTANCE (Top 10) ===")
print(builtin_importance.head(10).to_string(index=False))

# Visualize
plt.figure(figsize=(10, 8))
top_features = builtin_importance.head(15)
plt.barh(range(len(top_features)), top_features['importance'])
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Gini Importance')
plt.title('Top 15 Features by Built-in Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

**Reading the output:**

The table and horizontal bar chart rank the top 15 cell-nucleus features by built-in Gini importance. Features from the "worst" category — `worst_concave_points`, `worst_radius`, `worst_perimeter` — typically dominate because they capture the most extreme cell nuclei in each tissue sample, where malignancy signal concentrates. A steep dropoff after the top 3-5 features suggests that the forest's screening decisions are driven by a small subset of measurements.

The importance values sum to 1.0 across all 30 features. However, remember the caveat: Gini importance is biased toward continuous features with many unique split points, and it double-counts correlated features. `worst_radius` and `worst_perimeter` are highly correlated (r > 0.99) — they measure nearly the same thing (cell size) — so their combined Gini importance overstates the unique diagnostic information each provides.

**Why this matters for the lab directors:** If you took this chart at face value and invested in calibrating both `worst_radius` and `worst_perimeter` measurement equipment, you would be paying twice for essentially the same information. The next section introduces permutation importance, which avoids this double-counting trap.

---

## 6. Permutation Importance (Recommended)

Built-in Gini importance has a known flaw for the screening task: it inflates the apparent importance of correlated measurements like `worst_radius` and `worst_perimeter`, making it impossible to tell which cell measurement *uniquely* drives the diagnosis. **Permutation importance** avoids this by asking a cleaner question: "If I scramble the values of `worst_concave_points` across all test patients, how much does screening ROC-AUC drop?" If the answer is "a lot," that feature is genuinely important. If the answer is "barely at all," the forest can compensate using correlated features.

We compute permutation importance on the **test set** (not training set) so the estimates reflect genuine predictive value on unseen patients, not memorized patterns. Each feature is shuffled 10 times to produce a mean and standard deviation — essentially a confidence interval on how much each measurement matters for real-world screening performance.

> 💡 **Gemini Prompt:** "Compute permutation importance on test set (10 repeats, ROC-AUC, seed 474). Print top 10 and compare permutation vs Gini rankings side by side."
>
> **After running, verify:**
> - Permutation importance uses test data (not training)
> - Top 10 shows feature name, mean importance, std
> - Side-by-side comparison reveals ranking differences
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Permutation importance
perm_importance = permutation_importance(
    rf_final, X_test, y_test,
    n_repeats=10,
    random_state=RANDOM_SEED,
    scoring='roc_auc'
)

perm_df = pd.DataFrame({
    'feature': X.columns,
    'importance_mean': perm_importance.importances_mean,
    'importance_std': perm_importance.importances_std
}).sort_values('importance_mean', ascending=False)

print("=== PERMUTATION IMPORTANCE (Top 10) ===")
print(perm_df.head(10).to_string(index=False))

# Compare built-in vs permutation
comparison = builtin_importance.merge(
    perm_df,
    on='feature',
    suffixes=('_builtin', '_perm')
).head(10)

print("\n=== TOP 10: BUILT-IN VS PERMUTATION ===")
print(comparison[['feature', 'importance', 'importance_mean']].to_string(index=False))

**Reading the output:**

The permutation importance table ranks features by how much test-set ROC-AUC drops when each feature is shuffled. The `importance_std` column shows variability across 10 shuffle repeats — a feature with high mean importance but also high std should be interpreted cautiously (the diagnostic value fluctuates across different random shuffles of the test patients).

The comparison table at the bottom reveals the key insight: built-in Gini importance and permutation importance produce different rankings. `worst_radius` and `worst_perimeter` both rank high by Gini (because each gets used in many tree splits), but their permutation importance may be lower and closer together. Why? Shuffling `worst_radius` only partially degrades screening performance because `worst_perimeter` (still intact) carries nearly identical information. The forest compensates by leaning on the correlated feature.

Features with permutation importance near zero or negative — typically `mean_fractal_dimension`, `mean_symmetry` — are essentially irrelevant to the screening decision on new patients, even if they appeared in many tree splits during training.

**Why this matters for the Health Department:** Permutation importance tells the lab directors which cell measurements — `worst_concave_points`, `worst_radius`, `mean_concavity` — actually drive the diagnosis on unseen patients. These are the measurements worth standardizing across partner hospitals, investing in higher-precision equipment for, and double-checking when a borderline result comes through.

---

## 📝 PAUSE-AND-DO Exercise 2 (5 minutes)

**Task:** Compute permutation importance and write 3 interpretation bullets.

Already done above! Now analyze:

---

### YOUR INTERPRETATION:

**Bullet 1: Top Features**  
[Which features are most important? Why might this be?]

**Bullet 2: Differences**  
[How do built-in vs permutation importance differ?]

**Bullet 3: Business Insight**  
[What does this tell you about the prediction task?]

---

## 7. Comprehensive Model Comparison

Before concluding, we line up every model type the Health Department has built so far: Logistic Regression (linear baseline), a single tuned Decision Tree, and Random Forests with 100 and 200 trees. All four are evaluated under the same `StratifiedKFold` CV object — same folds, same patient partitions — so the comparison is fair and the numbers are directly comparable to those in the previous notebooks.

The resulting table and bar chart answer a question the board will ask: "How much does the Random Forest improve screening over the simpler models we already have?" Pay attention not only to the mean ROC-AUC but also to the standard deviation (stability across patient partitions) and the gap between CV and test scores (generalization). A model with both the highest mean and the lowest spread is the strongest candidate for statewide deployment.

> 💡 **Gemini Prompt:** "Compare four models -- LogisticRegression (pipeline), DecisionTree(depth=5), RandomForest(100), RandomForest(200) -- using 5-fold CV and test ROC-AUC. Print sorted comparison table and grouped bar chart."
>
> **After running, verify:**
> - Four models with CV mean/std and test scores
> - Table sorted by CV_Mean descending
> - Grouped bar chart has CV and Test bars per model
> - All numerical outputs use standard decimal format — no scientific notation


In [ ]:
# Compare all models
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(random_state=RANDOM_SEED, max_iter=1000))
    ]),
    'Decision Tree (tuned)': DecisionTreeClassifier(max_depth=5, random_state=RANDOM_SEED),
    'Random Forest (100)': RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1),
    'Random Forest (200)': RandomForestClassifier(n_estimators=200, random_state=RANDOM_SEED, n_jobs=-1)
}

comparison_results = []
for name, model in models.items():
    cv_results = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc')
    
    # Fit and test
    model.fit(X_train, y_train)
    if hasattr(model, 'predict_proba'):
        test_score = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
    else:
        test_score = roc_auc_score(y_test, model.predict(X_test))
    
    comparison_results.append({
        'Model': name,
        'CV_Mean': cv_results.mean(),
        'CV_Std': cv_results.std(),
        'Test_Score': test_score
    })

final_comparison = pd.DataFrame(comparison_results).sort_values('CV_Mean', ascending=False)
print("=== FINAL MODEL COMPARISON ===")
print(final_comparison.to_string(index=False))

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(final_comparison))
ax.bar(x - 0.2, final_comparison['CV_Mean'], 0.4, label='CV Mean', alpha=0.8)
ax.bar(x + 0.2, final_comparison['Test_Score'], 0.4, label='Test', alpha=0.8)
ax.set_xlabel('Model')
ax.set_ylabel('ROC-AUC')
ax.set_title('Model Comparison: CV vs Test Performance')
ax.set_xticks(x)
ax.set_xticklabels(final_comparison['Model'], rotation=45, ha='right')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

best_model = final_comparison.iloc[0]['Model']
print(f"\n✓ Champion model: {best_model}")

**Reading the output:**

The final comparison table ranks four models by CV ROC-AUC. Typical ordering on the breast cancer dataset: the two Random Forest variants lead with ROC-AUC around **0.99**, followed closely by Logistic Regression at **0.98-0.99**, with the single Decision Tree trailing at **0.96-0.97**. The 200-tree forest is usually within **0.001-0.003** of the 100-tree forest, illustrating the diminishing returns of adding more trees.

Notice the standard deviations: forests and logistic regression tend to have similarly low std (~0.01), while the single tree has noticeably higher std (~0.02-0.03). For the Health Department, this means the forest delivers *consistently* high screening accuracy across different patient partitions, whereas the single tree's performance fluctuates — unacceptable for a statewide deployment where every partner hospital expects reliable results.

The CV-vs-test gap should be small for all models. A large discrepancy would signal overfitting or an unusual test split. The champion model name is printed at the bottom. On this dataset the margin between the forest and logistic regression is often small, illustrating that a more complex model does not always provide a large lift over a well-tuned linear baseline.

**Key takeaway:** Random Forests reliably improve over single trees and compete with linear models on the screening dataset. The next notebook introduces Gradient Boosting — a sequential ensemble that may squeeze out additional performance by specifically targeting the cases the forest gets wrong.

---

## 8. Wrap-Up: Key Takeaways

### What We Learned Today:

1. **Bagging Reduces Variance**: Averaging many trees stabilizes predictions
2. **Random Forests**: Bagging + random feature selection = powerful ensemble
3. **Hyperparameter Tuning**: n_estimators and max_features most important
4. **OOB Score**: Free cross-validation estimate
5. **Permutation Importance**: More reliable than built-in Gini importance

### Critical Rules:

> **"More trees is almost always better (diminishing returns after 100-500)"**

> **"Use permutation importance for production, not built-in importance"**

> **"Random forests rarely overfit badly (but can underfit)"**

### Next Steps:

- Next notebook: Gradient Boosting (sequential ensembles)
- Boosting will give even better performance
- But requires more careful tuning

---

## Participation Assignment Submission Instructions

### To Submit This Notebook:

1. **Complete all exercises**: Fill in both PAUSE-AND-DO exercise cells with your findings
2. **Run All Cells**: Execute `Runtime → Run all` to ensure everything works
3. **Save a Copy**: `File → Save a copy in Drive or Download the .ipynb extension`
4. **Submit**: Upload your `.ipynb` file in the participation assignment you find in the course Brightspace page.

### Before Submitting, Check:

- [ ] All cells execute without errors
- [ ] All outputs are visible
- [ ] Both exercise responses are complete
- [ ] Notebook is shared with correct permissions
- [ ] You can explain every line of code you wrote

### Next Step:

Complete the **Quiz** in Brightspace (auto-graded)

---

## Bibliography

- Breiman, L. (2001). "Random Forests." *Machine Learning*, 45(1), 5-32.
- James, G., Witten, D., Hastie, T., & Tibshirani, R. (2021). *An Introduction to Statistical Learning with Python* - Tree-Based Methods (bagging/forests)
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* - Random forests and bagging
- scikit-learn User Guide: [RandomForest estimators](https://scikit-learn.org/stable/modules/ensemble.html#forest)
- scikit-learn User Guide: [Permutation importance](https://scikit-learn.org/stable/modules/permutation_importance.html)

---



<center>

Thank you!

</center>